In [14]:
"""Task 1: Trace the supplied tree by hand using Minimax (code prints the trace)."""
import math

def minimax(depth, idx, max_turn, scores, target, trace=True):
    pad = "  " * depth
    if depth == target:
        if trace: print(f"{pad}leaf[{idx}] = {scores[idx]}")
        return scores[idx]
    kind = "MAX" if max_turn else "MIN"
    if trace: print(f"{pad}{kind} node (depth {depth}, index {idx})")
    left = minimax(depth + 1, idx * 2, not max_turn, scores, target, trace)
    right = minimax(depth + 1, idx * 2 + 1, not max_turn, scores, target, trace)
    val = max(left, right) if max_turn else min(left, right)
    if trace: print(f"{pad}-> {kind}({left}, {right}) = {val}")
    return val

scores = [3, 5, 2, 9, 0, 7]
depth = math.ceil(math.log2(len(scores)))
scores += [0] * (2 ** depth - len(scores))      # pad to complete binary tree
print("Padded leaves:", scores, "| tree depth:", depth)
print("The optimal value is:", minimax(0, 0, True, scores, depth))

Padded leaves: [3, 5, 2, 9, 0, 7, 0, 0] | tree depth: 3
MAX node (depth 0, index 0)
  MIN node (depth 1, index 0)
    MAX node (depth 2, index 0)
      leaf[0] = 3
      leaf[1] = 5
    -> MAX(3, 5) = 5
    MAX node (depth 2, index 1)
      leaf[2] = 2
      leaf[3] = 9
    -> MAX(2, 9) = 9
  -> MIN(5, 9) = 5
  MIN node (depth 1, index 1)
    MAX node (depth 2, index 2)
      leaf[4] = 0
      leaf[5] = 7
    -> MAX(0, 7) = 7
    MAX node (depth 2, index 3)
      leaf[6] = 0
      leaf[7] = 0
    -> MAX(0, 0) = 0
  -> MIN(7, 0) = 0
-> MAX(5, 0) = 5
The optimal value is: 5


In [15]:
"""Task 2: Leaf-evaluation counter added to both Minimax and Alpha-Beta."""
import math

def minimax(depth, idx, max_turn, scores, target, cnt):
    if depth == target:
        cnt[0] += 1
        return scores[idx]
    f = max if max_turn else min
    return f(minimax(depth + 1, idx * 2 + i, not max_turn, scores, target, cnt) for i in range(2))

def alphabeta(depth, idx, max_turn, scores, target, alpha, beta, cnt):
    if depth == target:
        cnt[0] += 1
        return scores[idx]
    if max_turn:
        value = float('-inf')
        for i in range(2):
            value = max(value, alphabeta(depth + 1, idx * 2 + i, False, scores, target, alpha, beta, cnt))
            alpha = max(alpha, value)
            if alpha >= beta: break
        return value
    value = float('inf')
    for i in range(2):
        value = min(value, alphabeta(depth + 1, idx * 2 + i, True, scores, target, alpha, beta, cnt))
        beta = min(beta, value)
        if alpha >= beta: break
    return value

def compare(name, scores):
    s = list(scores)
    d = max(1, math.ceil(math.log2(len(s))))
    s += [0] * (2 ** d - len(s))
    c1, c2 = [0], [0]
    m = minimax(0, 0, True, s, d, c1)
    a = alphabeta(0, 0, True, s, d, float('-inf'), float('inf'), c2)
    print(f"{name:26s} leaves={s}\n{'':26s} minimax={m} (leaf evals={c1[0]}) | alphabeta={a} (leaf evals={c2[0]}) | same={m == a}")

if __name__ == "__main__":
    print("--- Supplied trees ---")
    compare("Minimax-lab tree", [3, 5, 2, 9, 0, 7])
    compare("Alpha-beta-lab tree", [3, 5, 2, 9, 3, 5, 2, 9])
    print("--- Edge cases ---")
    compare("All equal", [5] * 8)
    compare("All negative", [-3, -8, -1, -9, -4, -2, -7, -6])
    compare("Two leaves", [4, 9])
    compare("5 negative leaves (padding)", [-4, -1, -6, -2, -8])

--- Supplied trees ---
Minimax-lab tree           leaves=[3, 5, 2, 9, 0, 7, 0, 0]
                           minimax=5 (leaf evals=8) | alphabeta=5 (leaf evals=8) | same=True
Alpha-beta-lab tree        leaves=[3, 5, 2, 9, 3, 5, 2, 9]
                           minimax=5 (leaf evals=8) | alphabeta=5 (leaf evals=6) | same=True
--- Edge cases ---
All equal                  leaves=[5, 5, 5, 5, 5, 5, 5, 5]
                           minimax=5 (leaf evals=8) | alphabeta=5 (leaf evals=5) | same=True
All negative               leaves=[-3, -8, -1, -9, -4, -2, -7, -6]
                           minimax=-3 (leaf evals=8) | alphabeta=-3 (leaf evals=7) | same=True
Two leaves                 leaves=[4, 9]
                           minimax=9 (leaf evals=2) | alphabeta=9 (leaf evals=2) | same=True
5 negative leaves (padding) leaves=[-4, -1, -6, -2, -8, 0, 0, 0]
                           minimax=0 (leaf evals=8) | alphabeta=0 (leaf evals=7) | same=True


In [16]:
"""Task 3: Reorder branches; pruning changes but the Minimax value stays the same."""
import math, itertools, random

def minimax(depth, idx, max_turn, scores, target, cnt):
    if depth == target:
        cnt[0] += 1
        return scores[idx]
    f = max if max_turn else min
    return f(minimax(depth + 1, idx * 2 + i, not max_turn, scores, target, cnt) for i in range(2))

def alphabeta(depth, idx, max_turn, scores, target, alpha, beta, cnt):
    if depth == target:
        cnt[0] += 1
        return scores[idx]
    if max_turn:
        value = float('-inf')
        for i in range(2):
            value = max(value, alphabeta(depth + 1, idx * 2 + i, False, scores, target, alpha, beta, cnt))
            alpha = max(alpha, value)
            if alpha >= beta: break
        return value
    value = float('inf')
    for i in range(2):
        value = min(value, alphabeta(depth + 1, idx * 2 + i, True, scores, target, alpha, beta, cnt))
        beta = min(beta, value)
        if alpha >= beta: break
    return value

def run(leaves):
    d = int(math.log2(len(leaves)))
    c1, c2 = [0], [0]
    m = minimax(0, 0, True, leaves, d, c1)
    a = alphabeta(0, 0, True, leaves, d, float('-inf'), float('inf'), c2)
    return m, c1[0], a, c2[0]

base = [3, 5, 2, 9, 3, 5, 2, 9]
print("--- Different orderings of the same supplied tree ---")
orders = {
    "Original":       base,
    "Pairs swapped":  [5, 3, 9, 2, 5, 3, 9, 2],
    "Halves swapped": base[4:] + base[:4],
    "Ascending":      sorted(base),
    "Descending":     sorted(base, reverse=True),
}
for name, leaves in orders.items():
    m, n1, a, n2 = run(leaves)
    print(f"{name:15s} {leaves}  minimax={m} (evals {n1}) | alphabeta={a} (evals {n2}) | same={m == a}")

print("\n--- All 8! orderings of leaves 1..8 ---")
res = []
for p in itertools.permutations(range(1, 9)):
    m, n1, a, n2 = run(list(p))
    assert m == a
    res.append(n2)
print(f"alphabeta evals: min={min(res)}, max={max(res)}, mean={sum(res)/len(res):.2f}; minimax always 8")

print("\n--- 2000 random trees, 16 leaves ---")
random.seed(1)
agree = t1 = t2 = 0
for _ in range(2000):
    m, n1, a, n2 = run([random.randint(-20, 20) for _ in range(16)])
    agree += (m == a); t1 += n1; t2 += n2
print(f"values agree: {agree}/2000 | mean evals minimax={t1/2000:.1f}, alphabeta={t2/2000:.1f}")

--- Different orderings of the same supplied tree ---
Original        [3, 5, 2, 9, 3, 5, 2, 9]  minimax=5 (evals 8) | alphabeta=5 (evals 6) | same=True
Pairs swapped   [5, 3, 9, 2, 5, 3, 9, 2]  minimax=5 (evals 8) | alphabeta=5 (evals 5) | same=True
Halves swapped  [3, 5, 2, 9, 3, 5, 2, 9]  minimax=5 (evals 8) | alphabeta=5 (evals 6) | same=True
Ascending       [2, 2, 3, 3, 5, 5, 9, 9]  minimax=5 (evals 8) | alphabeta=5 (evals 6) | same=True
Descending      [9, 9, 5, 5, 3, 3, 2, 2]  minimax=5 (evals 8) | alphabeta=5 (evals 6) | same=True

--- All 8! orderings of leaves 1..8 ---
alphabeta evals: min=5, max=8, mean=6.85; minimax always 8

--- 2000 random trees, 16 leaves ---
values agree: 2000/2000 | mean evals minimax=16.0, alphabeta=11.8
